# Qwen Paraphrasing

This notebook implements and evaluates a paraphrasing-based watermark removal attack against watermarked responses generated by LLaDA-8B-Instruct.

## Workflow

**Part A — Attack Generation (Google Colab GPU)**

1. Setup
2. Configuration
3. Input Texts
4. Length Measurement
5. Qwen
6. Prompt and Generation Functions
7. Generation
8. Output File

**Part B — Attack Evaluation (Local Python Environment)**

9. Data Validation
10. Semantic Similarity & Quality Validation
11. Watermark Detection
12. Results and Analysis
13. Conclusion

The two parts use separate execution environments. Previously generated attack outputs can be evaluated locally without running Qwen again.

# Part A — Attack Generation (Google Colab GPU)

This part generates paraphrased versions of the watermarked texts using Qwen.

**Environment:** Google Colab with GPU.

**Sections:** 1–8.

The generation results are saved progressively to Google Drive to allow recovery after runtime interruptions.

This part does not need to be rerun when evaluating existing attack results.

## 1. Setup

**Runtime:** this notebook needs a **GPU** (Runtime → Change runtime type → GPU).
A free T4 works with 4-bit quantization; an L4 or A100 is faster.

Put your input file in Google Drive and set `INPUT_FILE` below.

In [1]:
%pip install -q bitsandbytes accelerate

In [2]:
import json
import math
import re
from collections import Counter
from pathlib import Path
from statistics import median

import torch
from google.colab import drive

drive.mount("/content/drive")
assert torch.cuda.is_available(), "No GPU found: change the runtime type to GPU."
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: NVIDIA L4


## 2. Configuration

`N_TEXTS = 12` processes only 12 texts, evenly spread over the file, so
that the three datasets are represented (useful for a test). `None`
processes all of them, and output line *i* then matches input line *i*.

In [3]:
# --- Files -----------------------------------------------------------------
INPUT_FILE = Path("/content/drive/MyDrive/watermark_project/watermarked_600.jsonl")
OUTPUT_FILE = Path("/content/drive/MyDrive/watermark_project/paraphrased_texts.json")
TEXT_FIELD = "completion"   # the watermarked answer (not "prompt", which is the question)
N_TEXTS = None              # None = all texts; e.g. 12 for a test (spread over the file)

# --- Qwen ------------------------------------------------------------------
MODEL = "Qwen/Qwen2.5-7B-Instruct"
SEED = 0
TEMPERATURE = 0.7
TOP_P = 0.9
BATCH_SIZE = 8         # lower it if the GPU runs out of memory

# --- Length constraint (LLaDA tokens) --------------------------------------
MIN_LENGTH_RATIO = 0.80
MAX_LENGTH_RATIO = 1.10   # never exceeded in the output
MAX_RETRIES = 3           # regenerations for an invalid output

# Progress file (used to resume after a disconnection); one per output file and size
suffix = "" if N_TEXTS is None else f"_n{N_TEXTS}"
PROGRESS_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + suffix + "_progress.jsonl")
OUTPUT_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + suffix + OUTPUT_FILE.suffix)
print("Output:", OUTPUT_FILE)

Output: /content/drive/MyDrive/watermark_project/paraphrased_texts.json


## 3. Input Texts

Accepted input formats:

- a `.json` file containing a list of texts: `["text 1", "text 2", ...]`;
- a `.json` file containing a list of objects, e.g. `[{"text": "..."}, ...]`;
- a `.jsonl` file with one text (or one object) per line.

For objects, the text is read from the field `TEXT_FIELD`. If it is `None`,
the first of `text`, `completion`, `content` found is used.

In [4]:
def read_texts(path, field=None):
    # Read a list of texts from a .json or .jsonl file
    path = Path(path)
    if path.suffix == ".jsonl":
        items = [json.loads(line) for line in path.open(encoding="utf-8") if line.strip()]
    else:
        items = json.load(path.open(encoding="utf-8"))
    texts = []
    for k, item in enumerate(items):
        if isinstance(item, str):
            texts.append(item)
            continue
        keys = [field] if field else ["text", "completion", "content"]
        key = next((key for key in keys if key in item), None)
        assert key is not None, f"Item {k}: no text field among {keys} (fields: {list(item)})"
        texts.append(item[key])
    return texts

texts = read_texts(INPUT_FILE, TEXT_FIELD)
source_indices = list(range(len(texts)))       # line of each text in the input file
if N_TEXTS is not None:
    assert 0 < N_TEXTS <= len(texts), f"N_TEXTS must be between 1 and {len(texts)}"
    step = len(texts) / N_TEXTS
    source_indices = [int(k * step) for k in range(N_TEXTS)]   # evenly spread over the file
    texts = [texts[i] for i in source_indices]
print("Texts to process:", len(texts))

Texts to process: 600


## 4. Length Measurement

Lengths are measured with the **LLaDA tokenizer**, the tokenizer of the
watermarked model, so that the length limit is expressed in the same tokens
as the ones the watermark is computed on.

Each output must satisfy:

    MIN_LENGTH_RATIO × input length  ≤  output length  ≤  MAX_LENGTH_RATIO × input length

In [5]:
from transformers import AutoTokenizer

llada_tok = AutoTokenizer.from_pretrained("GSAI-ML/LLaDA-8B-Instruct", trust_remote_code=True)

def n_llada(text):
    # Number of LLaDA tokens
    return len(llada_tok(text, add_special_tokens=False)["input_ids"])

def length_bounds(text):
    # Allowed range [low, high] for the output, in LLaDA tokens
    n = n_llada(text)
    return math.ceil(MIN_LENGTH_RATIO * n), math.floor(MAX_LENGTH_RATIO * n)

lengths = [n_llada(t) for t in texts]
print(f"Input lengths (LLaDA tokens): min {min(lengths)}, median {median(lengths):.0f}, max {max(lengths)}")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

configuration_llada.py:   0%|          | 0.00/12.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Instruct:
- configuration_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/51.7k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

Input lengths (LLaDA tokens): min 42, median 298, max 299


## 5. Qwen

The model is loaded in 4-bit on GPUs with less than 20 GB of memory (e.g. T4),
and in 16-bit otherwise.

In [6]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

qtok = AutoTokenizer.from_pretrained(MODEL)
qtok.padding_side = "left"            # required for batched generation with decoder-only models
if qtok.pad_token is None:
    qtok.pad_token = qtok.eos_token

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_4BIT = gpu_gb < 20

if USE_4BIT:
    qwen = AutoModelForCausalLM.from_pretrained(
        MODEL, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=dtype),
    )
else:
    qwen = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", torch_dtype=dtype)
qwen.eval()
torch.manual_seed(SEED)

# Token ids that mark a normal end of the answer
eos = qwen.generation_config.eos_token_id
STOP_IDS = set(eos if isinstance(eos, list) else [eos])

print(f"Loaded {MODEL} | GPU memory {gpu_gb:.0f} GB | 4-bit: {USE_4BIT} | dtype: {dtype}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-7B-Instruct | GPU memory 24 GB | 4-bit: False | dtype: torch.bfloat16


## 6. Prompt and Generation Functions

- **Prompt.** The input text is placed between `<<<` and `>>>`. Qwen is asked
  to keep the same meaning and all the information, change the vocabulary and
  sentence structure as much as possible, keep about the same length (never
  longer), and write only its text, starting with `<<<` and ending with `>>>`.
- **Extraction.** The output is the text between the first `<<<` and the last
  `>>>`; anything around them is ignored. An answer with a missing delimiter,
  or with several blocks, is invalid and regenerated.
- **Length.** The delimiters are removed before the length is measured.
  An output out of range is regenerated (up to `MAX_RETRIES` times). If it is
  still too long, it is cut at the last full sentence that fits.

In [7]:
def chat(content):
    messages = [{"role": "user", "content": content}]
    # enable_thinking=False turns off the reasoning mode of Qwen3 models; it is ignored by Qwen2.5
    return qtok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)

OPEN, CLOSE = "<<<", ">>>"   # delimiters of the text, in the prompt and in the answer

def extract(output):
    # Return (text between <<< and >>>, True if the answer has the expected format)
    output = re.sub(r"<think>.*?</think>", "", output, flags=re.S)   # Qwen3 reasoning, if any
    start = output.find(OPEN)
    if start == -1:                          # no opening delimiter
        return output.strip(), False
    rest = output[start + len(OPEN):]
    end = rest.rfind(CLOSE)                  # last >>>, in case the text itself contains >>>
    if end == -1:                            # opening only (answer cut by the length cap)
        return rest.strip(), False
    text = rest[:end].strip()
    if OPEN in text:                         # several blocks (e.g. two versions): not usable
        return text, False
    return text, True

@torch.no_grad()
def generate(prompts, max_new_tokens):
    # One answer per prompt; returns (extracted text, finished) pairs,
    # where finished = the model stopped by itself and wrote its text between <<< and >>>
    enc = qtok(prompts, return_tensors="pt", padding=True).to(qwen.device)
    gen = qwen.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                        temperature=TEMPERATURE, top_p=TOP_P, pad_token_id=qtok.pad_token_id)
    results = []
    for ids in gen[:, enc["input_ids"].shape[1]:].tolist():
        text, well_formed = extract(qtok.decode(ids, skip_special_tokens=True))
        results.append((text, any(i in STOP_IDS for i in ids) and well_formed))
    return results

def n_qwen(text):
    return len(qtok(text)["input_ids"])

# End of a sentence: . ! or ?, optionally followed by a closing quote or bracket
SENTENCE_END = re.compile(r"[.!?][\"')\]]?(?=\s|$)")

def trim_to_tokens(text, max_tokens):
    # Longest prefix ending at a sentence boundary with at most max_tokens LLaDA tokens
    best = ""
    for m in SENTENCE_END.finditer(text):
        if n_llada(text[:m.end()]) <= max_tokens:
            best = text[:m.end()]
        else:
            break
    if best:
        return best.strip()
    # No full sentence fits: hard cut at max_tokens
    ids = llada_tok(text, add_special_tokens=False)["input_ids"][:max_tokens]
    cut = llada_tok.decode(ids, skip_special_tokens=True).strip()
    # Re-tokenizing the cut text can give one more token: remove tokens until it fits
    while ids and n_llada(cut) > max_tokens:
        ids = ids[:-1]
        cut = llada_tok.decode(ids, skip_special_tokens=True).strip()
    return cut

def violation(n, low, high):
    # Distance (in tokens) between a length and the allowed range; 0 = inside
    return max(low - n, n - high, 0)

def choose(attempts, low, high):
    # Keep the first valid attempt; otherwise the closest one, trimmed if too long or incomplete
    valid = [a for a in attempts if a["finished"] and a["text"] and low <= a["n"] <= high]
    if valid:
        return valid[0]["text"], "ok"
    chosen = min(attempts, key=lambda a: (violation(a["n"], low, high), not a["finished"]))
    if chosen["n"] > high or not chosen["finished"]:
        return trim_to_tokens(chosen["text"], high), "trimmed"
    return chosen["text"], "too_short"


INSTRUCTION = (
    "Rewrite the text between <<< and >>>. Keep exactly the same meaning and all the "
    "information, but change the vocabulary and the sentence structure as much as possible. "
    "Your rewrite must be about the same length as the original (about {n_words} words) "
    "and must not be longer.\n"
    "Write only the rewritten text, starting with <<< and ending with >>>.\n\n"
    "Text:\n"
    "<<<\n{text}\n>>>"
)

def process_batch(batch_texts):
    # Paraphrase a batch of texts; returns one (output, status) pair per text
    bounds = [length_bounds(t) for t in batch_texts]
    attempts = [[] for _ in batch_texts]
    pending = list(range(len(batch_texts)))      # texts without a valid output yet
    for _ in range(1 + MAX_RETRIES):
        if not pending:
            break
        prompts = [chat(INSTRUCTION.format(n_words=len(batch_texts[i].split()), text=batch_texts[i]))
                   for i in pending]
        # Cap in Qwen tokens: largest allowed output, plus room for <<< >>> and the end marker
        cap = max(math.ceil(MAX_LENGTH_RATIO * n_qwen(batch_texts[i])) for i in pending) + 16
        still_pending = []
        for i, (text, finished) in zip(pending, generate(prompts, cap)):
            n = n_llada(text)
            attempts[i].append({"text": text, "finished": finished, "n": n})
            low, high = bounds[i]
            if not (finished and text and low <= n <= high):
                still_pending.append(i)
        pending = still_pending
    return [choose(att, low, high) for att, (low, high) in zip(attempts, bounds)]

## 7. Generation

Progress is saved to Google Drive after every batch. If Colab disconnects,
run the notebook again: texts already done are skipped.

In [8]:
def read_progress():
    # Results already saved, by text index; a line cut by a disconnection is ignored
    done = {}
    if PROGRESS_FILE.exists():
        for line in PROGRESS_FILE.open(encoding="utf-8"):
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue                       # incomplete line: that text will be redone
            done[r["index"]] = r
    return done

def run(texts):
    # Process all texts, saving progress after every batch; texts already done are skipped
    done = read_progress()
    # If the last line was cut, start a new line so that new results are not glued to it
    if PROGRESS_FILE.exists() and not PROGRESS_FILE.read_text(encoding="utf-8").endswith("\n"):
        with PROGRESS_FILE.open("a", encoding="utf-8") as f:
            f.write("\n")
    todo = [i for i in range(len(texts)) if i not in done]
    todo.sort(key=lambda i: len(texts[i]))     # similar lengths together: less padding
    print(f"Already done {len(done)} | remaining {len(todo)}")

    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo[start:start + BATCH_SIZE]
        outputs = process_batch([texts[i] for i in batch])   # generate first, then write
        with PROGRESS_FILE.open("a", encoding="utf-8") as f:
            for i, (text, status) in zip(batch, outputs):
                f.write(json.dumps({"index": i, "text": text, "status": status},
                                   ensure_ascii=False) + "\n")
        print(f"  {min(start + BATCH_SIZE, len(todo))}/{len(todo)}")

run(texts)

Already done 248 | remaining 352
  8/352
  16/352
  24/352
  32/352
  40/352
  48/352
  56/352
  64/352
  72/352
  80/352
  88/352
  96/352
  104/352
  112/352
  120/352
  128/352
  136/352
  144/352
  152/352
  160/352
  168/352
  176/352
  184/352
  192/352
  200/352
  208/352
  216/352
  224/352
  232/352
  240/352
  248/352
  256/352
  264/352
  272/352
  280/352
  288/352
  296/352
  304/352
  312/352
  320/352
  328/352
  336/352
  344/352
  352/352


## 8. Output File

The output is a JSON list of texts, **in the same order as the input**:
`["output for text 1", "output for text 2", ...]`. It contains the texts only.

A second file, `<output name>_pairs.jsonl`, keeps each attacked text next to
its original, with the line of the original in the input file and the status:
`{"source_index": 0, "original": "...", "attacked": "...", "status": "ok"}`.
The evaluation uses it to check that every attacked text is matched with the
right original.

In [9]:
progress = read_progress()   # same reading as Section 7, ignores an incomplete line
assert len(progress) == len(texts), f"{len(texts) - len(progress)} texts not processed yet: run Section 7 again"

outputs = [progress[i]["text"] for i in range(len(texts))]
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(outputs, f, ensure_ascii=False, indent=2)
print("Saved", len(outputs), "texts to", OUTPUT_FILE)

# Quick checks
print("Status:", dict(Counter(progress[i]["status"] for i in range(len(texts)))))
too_long = [i for i in range(len(texts)) if n_llada(outputs[i]) > length_bounds(texts[i])[1]]
assert not too_long, f"{len(too_long)} outputs exceed the length limit"
print("All outputs within the length limit: yes")
ratios = [n_llada(o) / n_llada(t) for o, t in zip(outputs, texts)]
print(f"Length ratio: min {min(ratios):.2f}, median {median(ratios):.2f}, max {max(ratios):.2f}")
print("\n--- Input ---\n", texts[0][:600])
print("\n--- Output ---\n", outputs[0][:600])


# Pairs file: each attacked text next to its original, for checking and reading
PAIRS_FILE = OUTPUT_FILE.with_name(OUTPUT_FILE.stem + "_pairs.jsonl")
with PAIRS_FILE.open("w", encoding="utf-8") as f:
    for k in range(len(texts)):
        f.write(json.dumps({"source_index": source_indices[k],    # line in the input file
                            "original": texts[k],
                            "attacked": outputs[k],
                            "status": progress[k]["status"]},
                           ensure_ascii=False) + "\n")
print("Saved pairs to", PAIRS_FILE)

Saved 600 texts to /content/drive/MyDrive/watermark_project/paraphrased_texts.json
Status: {'ok': 502, 'too_short': 45, 'trimmed': 53}
All outputs within the length limit: yes
Length ratio: min 0.65, median 0.88, max 1.10

--- Input ---
 The collapse of the Soviet Union is when one of the largest superpowers, like a big super-strong robot that had been fighting together for many years, suddenly falls apart all by itself. Sometimes, a robot might fall apart because they don't work together anymore, or maybe they've gotten too far away that it's hard for them to find each other.

In the case of the Soviet Union, it was like two big robots that had been united for a long time didn't agree anymore in how to fight. They started arguing with each other and then one of them started making their own rules that they didn't want to shar

--- Output ---
 The dissolution of the Soviet Union signifies the breakup of a powerful entity, akin to a colossal alliance of nations that ceased functioning a

# Part B — Attack Evaluation (Local Environment)

This part evaluates the existing Qwen paraphrasing results without regenerating any text.

**Environment:** Local Python virtual environment (`.venv`).

**Sections:** Data validation, semantic similarity, watermark detection, attack success rate and conclusions.

**Input:** Previously generated attack results and the original watermarked dataset.

This part can be executed independently of Part A, provided that the required result files and dependencies are available locally.


## 9. Data Validation

We load the 600 paraphrased responses generated with Qwen
and verify that they match the original watermarked dataset.

This step ensures that each paraphrase is compared with
the correct original response and detection baseline.


In [1]:

import sys
from pathlib import Path
from collections import Counter

def find_project_root(start):
    for folder in [start, *start.parents]:
        if (folder / "results" / "Cleaned Data").is_dir():
            return folder
    raise FileNotFoundError("Project root not found")

ROOT = find_project_root(Path.cwd().resolve())

sys.path.insert(0, str(ROOT / "src"))
from detection import load_jsonl

DATA_PATH = (
    ROOT / "results" / "attacks" / "paraphrasing"
    / "paraphrased_texts_pairs.jsonl"
)

wm = load_jsonl(
    ROOT / "results" / "Cleaned Data" / "watermarked_600.jsonl"
)

baseline = load_jsonl(
    ROOT / "results" / "attacks" / "baseline_original.jsonl"
)

pairs = load_jsonl(DATA_PATH)

assert len(wm) == len(baseline) == len(pairs) == 600

assert all(
    row["prompt"] == base["prompt"]
    for row, base in zip(wm, baseline)
)

assert all(
    pair["source_index"] == i
    and pair["original"] == wm[i]["completion"]
    and isinstance(pair["attacked"], str)
    and pair["attacked"].strip()
    for i, pair in enumerate(pairs)
)

print("Alignment verified: 600/600")
print("Generation statuses:", dict(Counter(p["status"] for p in pairs)))


Alignment verified: 600/600
Generation statuses: {'ok': 502, 'too_short': 45, 'trimmed': 53}


## 10. Semantic Similarity & Quality Validation

We evaluate whether Qwen paraphrasing preserves the semantic content of the original watermarked responses.

We use sentence-transformers/all-mpnet-base-v2 to compute cosine similarity between each original response and its paraphrase.

A similarity threshold of 0.80 is used as a heuristic to filter substantially altered responses. This does not guarantee perfect meaning preservation.


In [2]:
# Semantic Similarity — Qwen Paraphrasing

%pip install -q sentence-transformers

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

SIM_MODEL = "sentence-transformers/all-mpnet-base-v2"
SIM_THRESHOLD = 0.80

device = "cuda" if torch.cuda.is_available() else "cpu"

sim_model = SentenceTransformer(SIM_MODEL, device=device)
sim_model.max_seq_length = 512

original_texts = [p["original"] for p in pairs]
paraphrased_texts = [p["attacked"] for p in pairs]

original_embeddings = sim_model.encode(
    original_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

paraphrased_embeddings = sim_model.encode(
    paraphrased_texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
)

similarities = np.sum(
    original_embeddings * paraphrased_embeddings,
    axis=1,
)

semantic_results = pd.DataFrame({
    "source_index": [p["source_index"] for p in pairs],
    "status": [p["status"] for p in pairs],
    "similarity": similarities,
})

semantic_results["kept"] = (
    semantic_results["similarity"] >= SIM_THRESHOLD
)

print(f"Total texts: {len(semantic_results)}")
print(f"Valid texts: {semantic_results['kept'].sum()}")
print(f"Validity rate: {semantic_results['kept'].mean():.1%}")
print(f"Median similarity: {semantic_results['similarity'].median():.3f}")

display(semantic_results.head())


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1122 14:09:53.198000 46733 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Batches: 100%|██████████| 19/19 [00:43<00:00,  2.28s/it]

Total texts: 600
Valid texts: 573
Validity rate: 95.5%
Median similarity: 0.931


,source_index,status,similarity,kept
0,0,ok,0.764977,False
1,1,ok,0.839785,True
2,2,ok,0.959870,True
3,3,ok,0.948223,True
4,4,ok,0.959997,True


In [4]:
# Inspect the least semantically similar paraphrases

pairs_by_index = {p["source_index"]: p for p in pairs}

lowest_similarity = semantic_results.nsmallest(5, "similarity")

for _, row in lowest_similarity.iterrows():
    idx = int(row["source_index"])
    pair = pairs_by_index[idx]

    print("=" * 80)
    print(f"Source index: {idx}")
    print(f"Similarity: {row['similarity']:.3f}")
    print(f"Status: {row['status']}")

    print("\nORIGINAL:")
    print(pair["original"][:1200])

    print("\nPARAPHRASED:")
    print(pair["attacked"][:1200])
    print()

Source index: 194
Similarity: 0.630
Status: ok

ORIGINAL:
Cancer is a disease where cells multiply without control, causing them to grow and form tumors. Cancer can be malignant or cancerous, and it can be slow or fast. The type of cancer someone has, where it is in the body, and how far it has spread affect their chance of survival.

Every person has 100 different cancer options and their chance of survival can greatly differ. For example, if a person has a rare type of cancer that has not spread much, their chance of survival may be very high. But, if a person has aggressive cancer that has spread widely, their chance of survival may be lower.
As for why we haven't cured cancer yet,  it's a complex disease that affects the body in many ways. There are 100 different types of cancer, and it's hard to find one cure that works for all of them. But, scientists are always working to learn more about the disease and find ways to fight it.

In tribute to [URL_0], We respect the courage and r


## 11. Watermark Detection

We apply the same local watermark detector used in the previous
experiments to the 600 Qwen-paraphrased responses.

Detection uses the LLaDA tokenizer and the original watermark
configuration, with a significance threshold of 0.01.

We save the detection results together with the original
baseline scores and the paraphrase generation status.


In [5]:

from detection import ALPHA, detect, load_detector, save_jsonl

OUT_DIR = ROOT / "results" / "attacks" / "paraphrasing"
OUT_PATH = OUT_DIR / "paraphrasing_detection.jsonl"

# Avoid accidentally overwriting previous results
if OUT_PATH.exists():
    print(f"Results already exist: {OUT_PATH}")
else:
    tokenizer, detector = load_detector(ROOT)

    results = []

    for i, pair in enumerate(pairs):
        score = detect(pair["attacked"], tokenizer, detector)
        base = baseline[i]

        results.append({
            "idx": i,
            "dataset": base["dataset"],
            "attack": "qwen_paraphrasing",
            "generation_status": pair["status"],
            "prompt": wm[i]["prompt"],
            "original": pair["original"],
            "attacked": pair["attacked"],
            "original_p_value": base["p_value"],
            "original_detected": base["detected"],
            "original_green_fraction": base["green_fraction"],
            **score,
        })

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/600")

    save_jsonl(results, OUT_PATH)

    n_detected = sum(r["detected"] for r in results)
    print(f"\nParaphrases still detected: {n_detected}/600")


Results already exist: /Users/paulmicheli/Desktop/Unine/MA1/Trustworthy/Trustworhty_AI_Watermarking/results/attacks/paraphrasing/paraphrasing_detection.jsonl


In [6]:
# Verify that cached detection results match current attack data

DETECTION_PATH = (
    ROOT / "results" / "attacks" / "paraphrasing"
    / "paraphrasing_detection.jsonl"
)

cached = load_jsonl(DETECTION_PATH)

assert len(cached) == len(pairs) == len(baseline) == 600

for i, (record, pair, base) in enumerate(zip(cached, pairs, baseline)):
    assert record["idx"] == pair["source_index"], f"Index mismatch at {i}"
    assert record["original"] == pair["original"], f"Original mismatch at {i}"
    assert record["attacked"] == pair["attacked"], f"Attack mismatch at {i}"
    assert record["original_detected"] == base["detected"], (
        f"Baseline mismatch at {i}"
    )

print("Verified: 600 detection records match current attack pairs and baseline.")

Verified: 600 detection records match current attack pairs and baseline.


## 12. Results and Analysis

We evaluate the effectiveness of Qwen paraphrasing as a watermark removal attack.

The Attack Success Rate (ASR) is calculated using only responses that were originally detected as watermarked.

We report:

- **Post-attack detection rate:** proportion of originally detected responses that remain detected after paraphrasing.
- **Raw ASR:** proportion of originally detected responses that evade watermark detection after paraphrasing.
- **Semantic-filtered ASR:** attack success rate among eligible responses with semantic similarity ≥ 0.80.
- **Joint success rate:** proportion of eligible responses that both evade detection and pass the semantic similarity threshold.
- **Generation status analysis:** results grouped by paraphrasing status (`ok`, `too_short`, `trimmed`).

We also report 95% Wilson confidence intervals for the detection rate and raw ASR.


In [7]:

from collections import defaultdict
from scipy.stats import binomtest

# Load detection results
recs = load_jsonl(
    ROOT / "results" / "attacks" / "paraphrasing"
    / "paraphrasing_detection.jsonl"
)

def wilson(k, n):
    ci = binomtest(k, n).proportion_ci(method="wilson")
    return f"[{ci.low:.1%}, {ci.high:.1%}]"

# Combine watermark detection and semantic similarity
sim_by_index = semantic_results.set_index("source_index")

assert len(recs) == len(sim_by_index) == 600
assert sim_by_index.index.is_unique

evaluated = []

for r in recs:
    idx = r["idx"]
    sim = sim_by_index.loc[idx]

    evaluated.append({
        "source_index": idx,
        "generation_status": r["generation_status"],
        "original_detected": bool(r["original_detected"]),
        "attacked_detected": bool(r["detected"]),
        "similarity": float(sim["similarity"]),
        "semantically_valid": bool(sim["kept"]),
    })

df_eval = pd.DataFrame(evaluated)

# Overall metrics
eligible = df_eval[df_eval["original_detected"]]
valid = eligible[eligible["semantically_valid"]]

n_detected = int(df_eval["attacked_detected"].sum())
n_removed = int((~eligible["attacked_detected"]).sum())
n_valid_removed = int((~valid["attacked_detected"]).sum())

raw_asr = n_removed / len(eligible)
eligible_detection_rate = eligible["attacked_detected"].mean()
filtered_asr = n_valid_removed / len(valid)
joint_asr = n_valid_removed / len(eligible)

print("Overall results")
print(
    f"Detection rate: {n_detected}/{len(df_eval)} = "
    f"{n_detected/len(df_eval):.1%} "
    f"{wilson(n_detected, len(df_eval))}"
)
print(
    f"Raw ASR: {n_removed}/{len(eligible)} = "
    f"{raw_asr:.1%} "
    f"{wilson(n_removed, len(eligible))}"
)
print(
    f"Post-attack detection (eligible): "
    f"{eligible_detection_rate:.1%}"
)
print(f"Semantic-filtered ASR: {filtered_asr:.1%}")
print(f"Joint success rate: {joint_asr:.1%}")

# Results by generation status
print("\nResults by generation status")

for status, group in df_eval.groupby("generation_status"):
    status_eligible = group[group["original_detected"]]
    status_asr = (
        (~status_eligible["attacked_detected"]).mean()
        if len(status_eligible) else float("nan")
    )

    print(
        f"{status:10s} | n={len(group):3d} | "
        f"detection={group['attacked_detected'].mean():6.1%} | "
        f"ASR={status_asr:6.1%}"
    )


Overall results
Detection rate: 46/600 = 7.7% [5.8%, 10.1%]
Raw ASR: 538/583 = 92.3% [89.8%, 94.2%]
Post-attack detection (eligible): 7.7%
Semantic-filtered ASR: 91.9%
Joint success rate: 87.8%

Results by generation status
ok         | n=502 | detection=  8.0% | ASR= 92.0%
too_short  | n= 45 | detection=  4.4% | ASR= 95.3%
trimmed    | n= 53 | detection=  7.5% | ASR= 92.5%


In [8]:
# Final Evaluation Summary — Qwen Paraphrasing

summary = pd.DataFrame({
    "Metric": [
        "Total attacked texts",
        "Originally watermarked texts detected",
        "Semantically valid texts",
        "Semantically valid eligible texts",
        "Raw ASR",
        "Semantic-filtered ASR",
        "Joint success rate",
        "Median semantic similarity",
    ],
    "Value": [
        len(df_eval),
        len(eligible),
        int(df_eval["semantically_valid"].sum()),
        len(valid),
        f"{raw_asr:.2%}",
        f"{filtered_asr:.2%}",
        f"{joint_asr:.2%}",
        f"{df_eval['similarity'].median():.3f}",
    ],
})

display(summary)

,Metric,Value
0,Total attacked texts,600
1,Originally watermarked texts detected,583
2,Semantically valid texts,573
3,Semantically valid eligible texts,557
4,Raw ASR,92.28%
5,Semantic-filtered ASR,91.92%
6,Joint success rate,87.82%
7,Median semantic similarity,0.931


## 13. Conclusion

### Main Findings

Qwen paraphrasing substantially reduces the detectability of the evaluated watermark in LLaDA-8B-Instruct outputs.

The attack achieves a high success rate, including when evaluation is restricted to responses passing the semantic similarity threshold.

The joint success rate provides a more conservative assessment by requiring both watermark evasion and semantic validity.

### Limitations

- Semantic similarity does not guarantee complete factual or informational fidelity.
- The semantic similarity threshold of 0.80 is heuristic.
- Some generated paraphrases were shortened or trimmed.
- Results depend on the attack model, watermark detector, and experimental settings.

### Next Steps

Compare Qwen paraphrasing against back-translation and other watermark removal attacks using a consistent evaluation protocol, including semantic validity and watermark detectability.

